## Практика: эксперименты с векторными представлениями слов (всего 4 балла)

Сегодня будем работать с эмбеддингами слов: обучим небольшую собственную модель, загрузим предобученную модель из каталога gensim и применим её для визуализации текстовых корпусов.

В качестве исходных данных используем текстовый набор, на котором будем экспериментировать с векторными представлениями.

**Зависимости:** при локальном запуске потребуются `nltk`, `gensim`, `bokeh` (оригинальная команда: `pip install --upgrade nltk gensim bokeh`). Если используешь `uv`, установи зависимости через `uv add`.

**Этот ноутбук нужно сдать вместе с домашним заданием.**

> Неофициальный русский перевод [исходного семинара](./seminar.ipynb). Исполняемый код, тестовые проверки, имена переменных и структура ячеек сохранены.

In [ ]:
# загрузить исходные данные:
!wget "https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1" -O ./quora.txt
# альтернативная ссылка: https://yadi.sk/i/BPQrUu1NaTduEw

In [ ]:
import numpy as np

with open("./quora.txt", encoding="utf-8") as file:
    data = list(file)

data[50]

**Токенизация:** один из типичных первых шагов в NLP — разбить исходный текст на слова (токены).

Наши данные представлены необработанным текстом: знаки препинания и эмодзи могут примыкать к словам. Поэтому простого `str.split()` недостаточно.

Воспользуемся библиотекой **`nltk`**, которая решает многие задачи NLP: токенизацию, стемминг, определение частей речи и другие.

In [ ]:
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

In [ ]:
# ЗАДАНИЕ: переведи весь текст в нижний регистр и выдели токены через tokenizer. 
# data_tok — список списков токенов; один внутренний список на каждую строку data.

data_tok = []  # ТВОЙ КОД

In [ ]:
assert all(isinstance(row, (list, tuple)) for row in data_tok), "please convert each line into a list of tokens (strings)"
assert all(all(isinstance(tok, str) for tok in row) for row in data_tok), "please convert each line into a list of tokens (strings)"
is_latin = lambda tok: all('a' <= x.lower() <= 'z' for x in tok)
assert all(map(lambda l: not is_latin(l) or l.islower(), map(' '.join, data_tok))), "please make sure to lowercase the data"

In [ ]:
print([' '.join(row) for row in data_tok[:2]])

**Векторы слов:** получить эмбеддинги можно разными способами. Word2Vec и GloVe используют разные целевые функции, а fastText применяет представления на уровне символьных подсловных единиц.

Начнём с **gensim** — библиотеки для NLP с несколькими моделями векторных представлений, включая Word2Vec.

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(data_tok, 
                 vector_size=32,      # размерность эмбеддинга
                 min_count=5,         # учитывать слова, встретившиеся хотя бы пять раз
                 window=5).wv         # контекст — окно из пяти слов вокруг целевого

# Из документации gensim
# wv: объект, связывающий слова и их эмбеддинги.
# После обучения позволяет запрашивать векторы различными способами.

In [ ]:
# теперь можно получать векторы слов!
model.get_vector('anything')

In [ ]:
# или искать похожие слова напрямую. Поэкспериментируй!
model.most_similar('bread')

### Используем предобученную модель

Обучение собственной модели заняло какое-то время. А теперь представь обучение полноразмерных (100–300 измерений) эмбеддингов на гигабайтах текста — например, статьях Википедии или сообщениях Twitter.

К счастью, сегодня предобученные эмбеддинги можно загрузить буквально парой строк кода.

После первой загрузки модель кешируется в `~/gensim_data` (либо в `%USER_PATH%/gensim_data`). Путь к скачанным файлам можно проверить, установив параметр `return_path=True`.

In [ ]:
import gensim.downloader as api
model = api.load('glove-twitter-100')

In [ ]:
model.most_similar(positive=["coder", "money"], negative=["brain"])

```

```

```

```

```

```

```

```


# Визуализация данных с помощью эмбеддингов слов (1 балл)

Оценить качество наших векторов можно, изобразив их на графике. Проблема в том, что они живут в пространстве размерности от 30 и выше, а человеку привычнее два-три измерения.

К счастью, в машинном обучении есть методы **снижения размерности**.

Применим их к **1000 наиболее частотных словам**.

In [ ]:
words = model.index_to_key[:1000] 

print(words[::100])

In [ ]:
# получи вектор каждого слова с помощью model
word_vectors = []  # ТВОЙ КОД

In [ ]:
assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 100)
assert np.isfinite(word_vectors).all()

#### Линейная проекция: PCA (метод главных компонент)

Самый простой линейный метод снижения размерности — **Principal Component Analysis (PCA)**.

С геометрической точки зрения PCA ищет направления, вдоль которых данные обладают наибольшей дисперсией, то есть своего рода «естественные» оси.

<img src="https://github.com/yandexdataschool/Practical_RL/raw/master/yet_another_week/_resource/pca_fish.png" style="width:30%">

Метод пытается разложить матрицу «объекты × признаки» $X$ на две меньшие матрицы: $W$ и $\hat W$, минимизируя **среднеквадратичную ошибку** восстановления:

$$\|(X W) \hat{W} - X\|^2_2 \to_{W, \hat{W}} \min$$

- $X \in \mathbb{R}^{n \times m}$ — матрица объектов (**центрированная**);
- $W \in \mathbb{R}^{m \times d}$ — матрица прямого преобразования;
- $\hat{W} \in \mathbb{R}^{d \times m}$ — матрица обратного преобразования;
- $n$ — количество примеров, $m$ — исходная размерность, $d$ — целевая размерность.


In [ ]:
from sklearn.decomposition import PCA

# спроецируй векторы слов на плоскость с PCA, используя sklearn (fit, transform)
# затем нормализуй координаты до нулевого среднего и единичной дисперсии
word_vectors_pca = []  # ТВОЙ КОД

# при необходимости добавь код здесь

In [ ]:
assert word_vectors_pca.shape == (len(word_vectors), 2), "there must be a 2d vector for each word"
assert max(abs(word_vectors_pca.mean(0))) < 1e-5, "points must be zero-centered"
assert max(abs(1.0 - word_vectors_pca.std(0))) < 1e-2, "points must have unit variance"

#### Построим график!

> **Примечание о визуализации.** Эти графики служат для исследования данных. Интерактивные графики Bokeh зависят от среды Jupyter и иногда не появляются прямо внутри ноутбука, даже если координаты и объект графика корректны. Если числовые проверки `assert` проходят, а `draw_vectors` возвращает объект Bokeh, отсутствие встроенного графика не считается ошибкой. В качестве альтернативы можно построить статический scatter plot через Matplotlib или сохранить график Bokeh в HTML.

In [ ]:
import bokeh.models as bm, bokeh.plotting as pl
from bokeh.io import output_notebook
output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=600, height=400, show=True, **kwargs):
    """Строит интерактивный график точек с дополнительными данными при наведении."""
    if isinstance(color, str): color = [color] * len(x)
    data_source = bm.ColumnDataSource({ 'x' : x, 'y' : y, 'color': color, **kwargs })

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height)
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show:
        try:
            pl.show(fig)
        except Exception as error:
            print(f"Bokeh figure was created but could not be rendered inline: {error}")
    return fig

In [ ]:
draw_vectors(word_vectors_pca[:, 0], word_vectors_pca[:, 1], token=words)

# наведи курсор на точки и попробуй найти кластеры

#### Визуализируем ближайших соседей с помощью t-SNE

PCA полезен, но он строго линейный и поэтому хорошо отражает лишь общую структуру данных.

Если нам важнее сохранить локальные соседства (близкие точки должны оставаться близкими), можно использовать **t-SNE** — ещё один метод построения низкоразмерных представлений. Подробности: [как правильно интерпретировать t-SNE](https://distill.pub/2016/misread-tsne/).

In [ ]:
from sklearn.manifold import TSNE

# спроецируй векторы слов на плоскость через TSNE. Обучение может занять пару минут.
# нормализуй координаты так же, как после PCA


word_tsne = []  # ТВОЙ КОД

In [ ]:
assert isinstance(word_tsne, np.ndarray)
assert word_tsne.shape == (len(words), 2), "there must be a 2d vector for each word"
assert np.isfinite(word_tsne).all(), "t-SNE coordinates must be finite"

draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color='green', token=words)

### Визуализация фраз

Эмбеддинги слов позволяют представить вектором и короткую фразу. Самый простой подход — вычислить **среднее** векторов её токенов (при необходимости с весами).

Это помогает исследовать данные: находить выбросы, кластеры и другие особенности.

Попробуем применить такой подход к нашему корпусу!

In [ ]:
def get_phrase_embedding(phrase):
    """
    Преобразуй фразу в вектор, агрегируя эмбеддинги её слов. См. описание выше.
    """
    # 1. приведи фразу к нижнему регистру
    # 2. разбей фразу на токены
    # 3. усредни векторы токенов
    # пропускай слова, отсутствующие в словаре модели
    # если ни одно слово не известно модели, верни нулевой вектор
    
    vector = np.zeros([model.vector_size], dtype='float32')
    
    # ТВОЙ КОД
    
    return vector
        
    

In [ ]:
vector = get_phrase_embedding("I'm very sure. This never happened to me before...")

assert np.allclose(vector[::10],
                   np.array([ 0.31807372, -0.02558171,  0.0933293 , -0.1002182 , -1.0278689 ,
                             -0.16621883,  0.05083408,  0.17989802,  1.3701859 ,  0.08655966],
                              dtype=np.float32))
assert np.array_equal(get_phrase_embedding("thisisgibberish"), np.zeros([model.vector_size], dtype='float32')), "corner case for all missing words should be handled as described in the function comments"

In [ ]:
# для начала выберем ограниченный набор фраз.
chosen_phrases = data[::len(data) // 1000]

# вычисли векторы выбранных фраз
phrase_vectors = []  # ТВОЙ КОД

In [ ]:
assert isinstance(phrase_vectors, np.ndarray) and np.isfinite(phrase_vectors).all()
assert phrase_vectors.shape == (len(chosen_phrases), model.vector_size)

In [ ]:
# переведи векторы в 2D с помощью PCA, t-SNE или другого метода
# не забудь нормализовать

phrase_vectors_2d = TSNE().fit_transform(phrase_vectors)

phrase_vectors_2d = (phrase_vectors_2d - phrase_vectors_2d.mean(axis=0)) / phrase_vectors_2d.std(axis=0)

In [ ]:
draw_vectors(phrase_vectors_2d[:, 0], phrase_vectors_2d[:, 1],
             phrase=[phrase[:50] for phrase in chosen_phrases],
             radius=20,)

**Дальше идут новые задания. Продолжай прокручивать ноутбук.**
```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```


### Классификация текста: определение недопустимых комментариев

![img](https://github.com/yandexdataschool/nlp_course/raw/master/resources/banhammer.jpg)

**В этом ноутбуке** предстоит построить алгоритм, который различает обычные и токсичные комментарии из соцсетей.

Как и во многих реальных задачах, у нас есть лишь небольшой набор (около $10^3$ примеров), размеченный вручную. Сравним классические методы NLP с подходом на основе эмбеддингов.

In [ ]:
# если работаешь в Colab, скачай данные:
# !wget https://raw.githubusercontent.com/yandexdataschool/nlp_course/refs/heads/2025/week01_embeddings/comments.tsv -O ./comments.tsv -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
data = pd.read_csv("comments.tsv", sep='\t')

texts = data['comment_text'].values
target = data['should_ban'].values
data[50::200]

In [ ]:
from sklearn.model_selection import train_test_split
texts_train, texts_test, y_train, y_test = train_test_split(texts, target, test_size=0.5, random_state=42)

**Важно:** обычно данные следует разделять на обучающую и тестовую выборки **до** любой обработки.

Так мы защищаемся от утечки данных на этапе предобработки. Например, если ты выберешь в качестве признаков слова, встречающиеся в оскорбительных сообщениях, подсчитывать их нужно только в обучающей выборке. Иначе алгоритм получит информацию о тестовых данных, и оценка качества окажется завышенной.

### Предобработка и токенизация

Комментарии содержат необработанный текст: пунктуацию, буквы в разных регистрах, переводы строк и т. д.

Для последующих шагов разбей текст на токены с помощью одного из токенизаторов `nltk` и соедини их пробелами.

**Токенизатор нужно запускать ровно один раз.** После предобработки `texts_train` и `texts_test` — канонические строки с токенами, разделёнными пробелами, которые используются далее. Во всех следующих заданиях с BoW и TF-IDF получай токены через `text.split()`; **не применяй `TweetTokenizer`, `WordPunctTokenizer` или иной токенизатор повторно**.

In [ ]:
from nltk.tokenize import TweetTokenizer
tokenizer = TweetTokenizer()
preprocess = lambda text: ' '.join(tokenizer.tokenize(text.lower()))

text = 'How to be a grown-up at work: replace "fuck you" with "Ok, great!".'
print("before:", text,)
print("after:", preprocess(text),)

In [ ]:
# задание: выполни предобработку всех комментариев из train и test

texts_train = []  # ТВОЙ КОД
texts_test = []  # ТВОЙ КОД

In [ ]:
assert texts_train[5] ==  'who cares anymore . they attack with impunity .'
assert texts_test[89] == 'hey todds ! quick q ? why are you so gay'
assert len(texts_test) == len(y_test)

### Решение: мешок слов (Bag of Words, 1 балл)

![img](http://www.novuslight.com/uploads/n/BagofWords.jpg)

Традиционный подход — использовать признаки **Bag of Words (BoW)**:

1. Собрать словарь наиболее частотных слов (**только по обучающей выборке**).
2. Для каждого примера подсчитать, сколько раз встречается каждое слово из словаря.
3. Использовать эти количества как признаки классификатора.

`bow_vocabulary` должен быть **`list[str]`** из уникальных токенов, отсортированных по убыванию частоты в корпусе. `bow_vocabulary[j]` — токен, соответствующий столбцу признаков `j`. При равной частоте допускается любой детерминированный порядок. Множество `set` не подходит, поскольку не определяет индексы столбцов. Для ускорения поиска можно дополнительно создать словарь вида `token_to_id = {token: j for j, token in enumerate(bow_vocabulary)}`.

**Примечание:** в реальных проектах такие признаки обычно вычисляют средствами sklearn. Но **в этом задании** готовые векторизаторы применять нельзя:
- `from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer`

In [ ]:
# задание: найди не более k самых частотных токенов в texts_train,
# отсортируй по убыванию числа вхождений.
# bow_vocabulary должен быть списком: позиция токена определяет столбец признака.
# Здесь и в text_to_bow используй text.split(); повторная токенизация запрещена.
k = 10000

# ТВОЙ КОД

bow_vocabulary = []  # ТВОЙ КОД

print('example features:', sorted(bow_vocabulary)[::100])

In [ ]:
def text_to_bow(text):
    """Преобразуй текст в массив частот токенов на основе bow_vocabulary."""
    # ТВОЙ КОД
    
    return np.array([], 'float32')

In [ ]:
X_train_bow = np.stack(list(map(text_to_bow, texts_train)))
X_test_bow = np.stack(list(map(text_to_bow, texts_test)))

In [ ]:
k_max = len(set(' '.join(texts_train).split()))
bow_vocabulary_set = set(bow_vocabulary)
expected_counts = np.array([
    sum(token in bow_vocabulary_set for token in text.split())
    for text in texts_train[5:10]
])

assert X_train_bow.shape == (len(texts_train), min(k, k_max))
assert X_test_bow.shape == (len(texts_test), min(k, k_max))
assert np.all(X_train_bow[5:10].sum(-1) == expected_counts)
assert isinstance(bow_vocabulary, list), "bow_vocabulary must be a list whose positions define feature columns"
assert len(bow_vocabulary) <= min(k, k_max)
assert len(bow_vocabulary) == len(bow_vocabulary_set), "vocabulary tokens must be unique"
assert X_train_bow[6, bow_vocabulary.index('.')] == texts_train[6].split().count('.')

Будем использовать простую линейную модель — **логистическую регрессию (Logistic Regression)**.

Возможно, она не самая сложная, но у её простоты есть преимущества: обучение очень быстрое и требует сравнительно немного данных.

Обучим классификатор средствами sklearn.

In [ ]:
from sklearn.linear_model import LogisticRegression
bow_model = None  # ТВОЙ КОД — обучи логистическую регрессию

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

for name, X, y, model in [
    ('train', X_train_bow, y_train, bow_model),
    ('test ', X_test_bow, y_test, bow_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

test_accuracy = np.mean(bow_model.predict(X_test_bow) == y_test)
print(f"Model accuracy: {test_accuracy:.3f}")
assert test_accuracy > 0.77, "Hint: tune the parameter C to improve performance"
print("Well done!")

### Задание: самостоятельно реализовать TF-IDF (1 балл)

Не все слова одинаково информативны. Редким словам можно придавать больший вес, а частым служебным словам вроде «and» и «or» — меньший. Для этого используются **TF-IDF-признаки** (term frequency / inverse document frequency — частота термина / обратная частота документа):

$$ \text{feature}_i = \frac{\text{Count}(word_i \in x)}{\text{Total number of words in } x} \times \log\left(\frac{N}{\text{Count}(word_i \in D) + \alpha}\right) $$

Здесь $x$ — один текст, $D$ — весь датасет (множество текстов), $N$ — число документов, а $\alpha$ — параметр сглаживания (обычно 1). Величина $Count(word_i \in D)$ означает **число документов**, в которых встречается слово $word_i$.

После вычисления TF-IDF может быть полезно нормализовать вектор каждого примера.

**Твоя задача:** реализовать TF-IDF, обучить модель, построить ROC-кривую и сравнить результат с моделью Bag of Words из предыдущей части.

TF-IDF **не обязан** показывать качество выше, чем BoW, на этом датасете. Более низкие ROC-AUC или accuracy допустимы, если реализация корректна: порядок моделей по качеству не оценивается.

**Не используй** готовые TF-IDF-векторизаторы sklearn или nltk при реализации. Для отладки использовать их разрешается.

Когда закончишь с TF-IDF, прокрути вниз.
```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```


### Улучшаем решение: векторы слов (1 балл)

Попробуем другой подход: вместо подсчёта частот каждого слова заменим слова **предобученными эмбеддингами**, а затем усредним векторы слов текста, получив его признаки.

У этого подхода есть два основных преимущества: (1) теперь у нас порядка $10^2$ признаков вместо $10^4$ и (2) модель может обобщать знания на слова, которых не было в обучающей выборке.

Начнём со стандартного подхода с предобученными эмбеддингами. Также можно попробовать:

- Обучить эмбеддинги с нуля на подходящих неразмеченных данных.
- Умножать векторы слов на величину, обратную частоте слова в датасете (как в TF-IDF).
- Конкатенировать несколько эмбеддингов.
  - Посмотреть перечень доступных моделей: `gensim.downloader.info()['models'].keys()`.
- Кластеризовать слова по векторам и использовать «мешок идентификаторов кластеров».

**Примечание:** загрузка предобученной модели может занять заметное время.

In [ ]:
import gensim.downloader 
embeddings = gensim.downloader.load("fasttext-wiki-news-subwords-300")

# Если мало оперативной памяти или низкая скорость сети, используй "glove-wiki-gigaword-100". Тогда следующие assert-проверки не применимы.

In [ ]:
def vectorize_sum(comment):
    """
    Реализуй функцию, превращающую обработанный комментарий в сумму векторов токенов.
    """
    embedding_dim = embeddings.vectors.shape[1]
    features = np.zeros([embedding_dim], dtype='float32')
    
    # ТВОЙ КОД
    
    return features

assert np.allclose(
    vectorize_sum("who cares anymore . they attack with impunity .")[::70],
    np.array([ 0.0108616 ,  0.0261663 ,  0.13855131, -0.18510573, -0.46380025])
)

In [ ]:
X_train_wv = np.stack([vectorize_sum(text) for text in texts_train])
X_test_wv = np.stack([vectorize_sum(text) for text in texts_test])

In [ ]:
wv_model = LogisticRegression().fit(X_train_wv, y_train)

for name, X, y, model in [
    ('bow train', X_train_bow, y_train, bow_model),
    ('bow test ', X_test_bow, y_test, bow_model),
    ('vec train', X_train_wv, y_train, wv_model),
    ('vec test ', X_test_wv, y_test, wv_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

assert roc_auc_score(y_test, wv_model.predict_proba(X_test_wv)[:, 1]) > 0.92, "something's wrong with your features"

Если всё выполнено правильно, доля ошибок классификации должна уменьшиться примерно вдвое.

Такой приём полезен для маленьких датасетов. Если примеров сотни тысяч, есть и другие подходы; к ним перейдём во второй части.

**Дополнительные материалы:**
- Посмотри другие модели эмбеддингов: `gensim.downloader.info()`.
- Изучи [FastText](https://github.com/facebookresearch/fastText).